# 5.2 Apoptosis Modelling: BCL2 Delays Cell Death (Figure 1B)

In this practical, we build and run the **59-species Apoptosis ODE model** from *Norris et al. (Blood Cancer Journal, 2024)* to:

1. Recreate **Figure 1B (top)**: Compare **WT** vs **BCL2 × 1.5** `% Cleaved PARP` over time and watch the sharp, all-or-nothing cell-death switch!
2. Recreate **Figure 1B (bottom)**: Sweep **BCL2 overexpression from 1× to 100×** and see how high BCL2 delays—and eventually blocks—apoptosis!
3. **Rescue Exercise**: Screen candidate pro- and anti-apoptotic gene targets to see which over- or under-expressions can **rescue BCL2-mediated resistance to cell death**.

Click **⚡ Activate Live Editable Julia Cells** above to run and edit the simulations live!

## The Apoptosis Pathway Recap

![Apoptosis signalling circuit](../images/apoptosis-pathway.svg)

We measure **Time to Cell Death** as the time (in hours) when **Cleaved PARP (`CPARP`)** crosses **10%** of total PARP (`PARP + PARP_C3 + CPARP`).

## Step 1: Load Packages & Compile the 59-Species Apoptosis Model

We load **only** the four precompiled packages (`DifferentialEquations`, `Plots`, `CSV`, `DataFrames`), compile the apoptosis CSV reaction network in `"scan"` mode (so every parameter can be adjusted dynamically via `modify[...]`), and apply `fixSpecies` so the external death-ligand input `L` (species `1`) stays clamped at `0` during steady-state equilibration and `1` during death-ligand treatment.

In [ ]:
if isdir("/home/jovyan/work/apoptosis")
    cd("/home/jovyan/work/apoptosis")
elseif isdir("apoptosis")
    cd("apoptosis")
end
@assert isfile("reactions_apoptosis.csv") "Apoptosis model files not found in $(pwd())."

using DifferentialEquations
using Plots
using CSV
using DataFrames

# Generate the 59-species ODE model in "scan" mode so any gene/parameter can be modified live
run(`python3 csv2model-multiscale.py reactions_apoptosis.csv parameters_apoptosis.csv rateLaws_apoptosis.csv odeApoptosis.jl scan`)

include("fixSpecies.jl")
fixSpecies("odeApoptosis.jl", "odeApoptosisFixed.jl", 1)   # Fix species 1 (death ligand L)
odeApopFun = include("odeApoptosisFixed.jl")
include("variableNames.jl")
include("scanIncludes.jl")

println("Apoptosis model ready: ", length(syms), " species, ", length(parameterList), " parameters.")

Apoptosis model ready: 59 species, 117 parameters.

## Step 2: Define Initial Conditions & Two-Phase Simulation Helper

Just like our NF-κB model, each virtual cell runs in **two phases**:
1. **Steady State (Phase 1):** Death ligand `L = 0`, with `Bcl2t = 277 × bcl2_mult` and any gene modifications in `mods`, equilibrated to resting steady state.
2. **Death-Ligand Time Course (Phase 2):** Switch `L = 1` and simulate for **144 hours** (6 days), tracking `% Cleaved PARP` every minute!

In [ ]:
const f_apop = ODEFunction(odeApopFun)
const maxTimeSS_apop = 100000.0
const maxTimeTC_apop = 144.0 * 60.0

const idx_L     = findfirst(isequal("L"), syms)
const idx_Bcl2t = findfirst(isequal("Bcl2t"), syms)
const idx_CPARP = findfirst(isequal("CPARP"), syms)
const idx_PARP_denom = [findfirst(isequal(s), syms) for s in ["PARP", "PARP_C3", "CPARP"]]

const apop_alg = if isdefined(Main, :AutoFiniteDiff)
    Rosenbrock23(autodiff=AutoFiniteDiff())
elseif isdefined(DifferentialEquations, :AutoFiniteDiff)
    Rosenbrock23(autodiff=DifferentialEquations.AutoFiniteDiff())
else
    Rosenbrock23()
end

function initConditionsApop(syms)
    # Initial molecule counts from Albeck et al. (2008, PLoS Biology) & Norris et al. (2024)
    y0 = zeros(length(syms))
    inits = Dict(
        "L" => 0.0, "R" => 1000.0, "flip" => 2000.0, "pC8" => 10000.0,
        "BAR" => 1000.0, "pC3" => 10000.0, "pC6" => 10000.0, "XIAP" => 100000.0,
        "PARP" => 1000000.0, "Bid" => 60000.0, "Bcl2c" => 20000.0, "Bax" => 80000.0,
        "Bcl2" => 30000.0, "Mito" => 500000.0, "mCytoC" => 500000.0, "mSMac" => 100000.0,
        "Apaf" => 100000.0, "Procasp9" => 100000.0, "Bcl2t" => 277.0,
    )
    for (k, v) in inits
        idx = findfirst(isequal(k), syms)
        if !isnothing(idx)
            y0[idx] = v
        end
    end
    return y0
end

function run_apoptosis(bcl2_mult::Real; mods::Dict{String, <:Real}=Dict{String, Float64}())
    # Reset all parameter multipliers to 1.0, then apply requested modifications
    for k in keys(modify)
        modify[k] = 1.0
    end
    for (k, v) in mods
        if !haskey(modify, k)
            error("Parameter '$(k)' not found! Check the Apoptosis Target Genes table.")
        end
        modify[k] = Float64(v)
    end

    # Phase 1: Resting steady state (L = 0)
    y0 = initConditionsApop(syms)
    y0[idx_L] = 0.0
    y0[idx_Bcl2t] = 277.0 * Float64(bcl2_mult)
    prob_ss = ODEProblem(f_apop, y0, (0.0, maxTimeSS_apop))
    sol_ss  = solve(prob_ss, apop_alg; abstol=1e-5, reltol=1e-3, maxiters=10^7, save_everystep=false)

    # Phase 2: Death-ligand stimulation (L = 1) for 144 hours
    y0_tc = max.(collect(Float64, sol_ss.u[end]), 0.0)
    y0_tc[idx_L] = 1.0
    prob_tc = ODEProblem(f_apop, y0_tc, (0.0, maxTimeTC_apop))
    sol_tc  = solve(prob_tc, apop_alg; abstol=1e-5, reltol=1e-3, maxiters=10^7, saveat=1.0)

    sol_mat = Float64.(reduce(hcat, sol_tc.u))
    cparp   = sol_mat[idx_CPARP, :]
    total_p = max.(vec(sum(sol_mat[idx_PARP_denom, :], dims=1)), 1e-12)
    cparp_pct = 100.0 .* (cparp ./ total_p)
    t_hours   = sol_tc.t ./ 60.0

    # Time to cell death: first time Cleaved PARP crosses 10%
    death_idx = findfirst(x -> x >= 10.0, cparp_pct)
    survived  = isnothing(death_idx)
    t2d_hours = survived ? (maxTimeTC_apop / 60.0) : t_hours[death_idx]

    return (t_hours=t_hours, cparp_pct=cparp_pct, t2d_hours=t2d_hours, survived=survived)
end

println("Helper run_apoptosis(bcl2_mult; mods=Dict(...)) ready!")

Helper run_apoptosis(bcl2_mult; mods=Dict(...)) ready!

## Step 3: Recreate Figure 1B (Top) — WT vs BCL2 × 1.5

Run the cell below to simulate a **Wild-Type (WT)** B-cell alongside a **BCL2 × 1.5** B-cell.
Notice two striking biological features:
1. **Switch-like death:** Cleaved PARP sits at ~0% for dozens of hours, then shoots up almost vertically once MOMP triggers!
2. **Delayed death:** Just a **1.5× increase in BCL2** delays cell death by ~15 hours!

In [ ]:
res_wt   = run_apoptosis(1.0)
res_bcl2 = run_apoptosis(1.5)

println("WT time to cell death:        ", round(res_wt.t2d_hours, digits=1), " hours")
println("BCL2 x 1.5 time to cell death: ", round(res_bcl2.t2d_hours, digits=1), " hours")

gr()
p_top = plot(res_wt.t_hours, res_wt.cparp_pct, lw=3, color=:steelblue, label="WT ($(round(res_wt.t2d_hours, digits=1)) h)")
plot!(p_top, res_bcl2.t_hours, res_bcl2.cparp_pct, lw=3, color=:orangered, label="BCL2 × 1.5 ($(round(res_bcl2.t2d_hours, digits=1)) h)")
hline!(p_top, [10.0], linestyle=:dash, color=:gray, lw=1.5, label="10% death threshold")
plot!(p_top, xlabel="Time (h)", ylabel="Cleaved PARP (%)",
      title="Figure 1B (Top): Apoptosis — WT vs BCL2 × 1.5",
      xlims=(0, 144), ylims=(0, 70), grid=true,
      xtickfontsize=11, ytickfontsize=11, xguidefontsize=13, yguidefontsize=13, legendfontsize=10)
display(p_top)

## Step 4: Recreate Figure 1B (Bottom) — BCL2 Dose Sweep (1× to 100×)

Now let's sweep **BCL2 overexpression** across `[1, 1.5, 2, 5, 10, 50, 100]` and recreate the horizontal bar chart from **Figure 1B**!
Notice what happens once BCL2 reaches **10× or higher**: the cell resists apoptosis across the entire 144-hour simulation!

In [ ]:
bcl2_mults = [1.0, 1.5, 2.0, 5.0, 10.0, 50.0, 100.0]
t2d_hours  = Float64[]
survived_flags = Bool[]

for m in bcl2_mults
    r = run_apoptosis(m)
    push!(t2d_hours, r.t2d_hours)
    push!(survived_flags, r.survived)
end

summary_df = DataFrame(
    BCL2_Multiplier = bcl2_mults,
    Time_to_Death_h = round.(t2d_hours, digits=1),
    Status          = [s ? "Survived (>144 h)" : "Died" for s in survived_flags],
)
display(summary_df)

cA = palette(:seaborn_colorblind)
labels_y = ["1", "1.5", "2", "5", "10", "50", "100"]
bar_colors = [cA[1], cA[2], cA[3], cA[4], cA[5], cA[6], cA[8]]

p_bot = bar(
    1:length(bcl2_mults), t2d_hours;
    orientation=:horizontal,
    yflip=true,
    yticks=(1:length(bcl2_mults), labels_y),
    fillcolor=bar_colors,
    linecolor=:transparent,
    legend=false,
    xlims=(0, 148),
    xticks=0:20:140,
    xlabel="Time to cell death (h)",
    ylabel="BCL2 Multiplier",
    title="Figure 1B (Bottom): BCL2 Overexpression Delays & Blocks Apoptosis",
    size=(780, 340),
    xtickfontsize=11, ytickfontsize=11, xguidefontsize=13, yguidefontsize=13,
)
display(p_bot)

## Apoptosis Rescue Targets Reference Table

Can we **rescue** BCL2-overexpressing lymphoma cells (restoring WT-like apoptosis) by **over-expressing** a pro-apoptotic gene (`> 1.0`) or **under-expressing / inhibiting** an anti-apoptotic gene (`< 1.0`)?

| Gene / Protein | Role in Apoptosis | Parameter Name in Julia | To Promote Cell Death (Rescue BCL2) |
| :--- | :--- | :--- | :--- |
| **BAX** | Pro-apoptotic mitochondrial pore former | `"k1_86"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **BID** | Pro-apoptotic BH3-only activator (`tBid`) | `"k1_81"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **BCL2 (mitochondrial)** | Anti-apoptotic MOMP inhibitor *(venetoclax target!)* | `"k1_88"` | **Under-express / inhibit** (e.g. `0.5` or `0.3`) |
| **Bcl2c (cytosolic)** | Anti-apoptotic cytosolic `tBid` buffer | `"k1_83"` | **Under-express / inhibit** (e.g. `0.1` – `0.5`) |
| **XIAP** | Anti-apoptotic Caspase-3/9 inhibitor *(SMAC mimetic target)* | `"k1_95"` | **Under-express / inhibit** (e.g. `0.1` or `0.05`) |
| **SMAC (`mSMac`)** | Pro-apoptotic XIAP antagonist | `"k1_97"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **Pro-Caspase-3 (`pC3`)** | Executioner caspase that cleaves PARP | `"k1_101"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **FLIP** | Anti-apoptotic Caspase-8 inhibitor at DISC | `"k1_73"` | **Under-express / inhibit** (e.g. `0.1` – `0.5`) |

## Exercise: Rescue BCL2 Overexpression (`BCL2 × 2.0`)

In a `BCL2 × 2.0` lymphoma cell, apoptosis is delayed from ~61 hours to ~83 hours (and at `BCL2 × 10.0`, the cell never dies!).

**Your task:** Edit the `rescue_mods` dictionary in the starter cell below to test whether over-expressing or under-expressing a gene from the reference table can **bring `BCL2 × 2.0` back down to WT-like cell death (~61 hours)**!
*(Bonus: Once you find a rescue for `BCL2 × 2.0`, change `test_bcl2 = 10.0` and see if your target can even rescue a completely apoptosis-resistant `BCL2 × 10.0` cell!)*

````{admonition} 📋 Quick Reference: Apoptosis Target Genes
:class: note, dropdown

| Gene / Protein | Role in Apoptosis | Parameter Name in Julia | To Promote Cell Death (Rescue BCL2) |
| :--- | :--- | :--- | :--- |
| **BAX** | Pro-apoptotic mitochondrial pore former | `"k1_86"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **BID** | Pro-apoptotic BH3-only activator (`tBid`) | `"k1_81"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **BCL2 (mitochondrial)** | Anti-apoptotic MOMP inhibitor *(venetoclax target!)* | `"k1_88"` | **Under-express / inhibit** (e.g. `0.5` or `0.3`) |
| **Bcl2c (cytosolic)** | Anti-apoptotic cytosolic `tBid` buffer | `"k1_83"` | **Under-express / inhibit** (e.g. `0.1` – `0.5`) |
| **XIAP** | Anti-apoptotic Caspase-3/9 inhibitor *(SMAC mimetic target)* | `"k1_95"` | **Under-express / inhibit** (e.g. `0.1` or `0.05`) |
| **SMAC (`mSMac`)** | Pro-apoptotic XIAP antagonist | `"k1_97"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **Pro-Caspase-3 (`pC3`)** | Executioner caspase that cleaves PARP | `"k1_101"` | **Over-express** (e.g. `2.0` – `5.0`) |
| **FLIP** | Anti-apoptotic Caspase-8 inhibitor at DISC | `"k1_73"` | **Under-express / inhibit** (e.g. `0.1` – `0.5`) |
````

In [ ]:
# Exercise starter code — edit rescue_mods below to rescue BCL2 x 2.0 back toward WT!
test_bcl2   = 2.0
rescue_name = "BCL2 × 2.0 + My Rescue"
rescue_mods = Dict{String, Float64}()   # <-- Try e.g. Dict("k1_86" => 2.5) or Dict("k1_95" => 0.1)!

res_wt     = run_apoptosis(1.0)
res_mut    = run_apoptosis(test_bcl2)
res_rescue = run_apoptosis(test_bcl2; mods=rescue_mods)

display(DataFrame(
    Condition       = ["WT", "BCL2 × $(test_bcl2) (untreated)", rescue_name],
    Time_to_Death_h = round.([res_wt.t2d_hours, res_mut.t2d_hours, res_rescue.t2d_hours], digits=1),
    Status          = [r.survived ? "Survived (>144 h)" : "Died" for r in (res_wt, res_mut, res_rescue)],
))

p_ex = plot(res_wt.t_hours, res_wt.cparp_pct, lw=3, color=:steelblue, label="WT ($(round(res_wt.t2d_hours, digits=1)) h)")
plot!(p_ex, res_mut.t_hours, res_mut.cparp_pct, lw=3, color=:orangered, label="BCL2 × $(test_bcl2) ($(round(res_mut.t2d_hours, digits=1)) h)")
plot!(p_ex, res_rescue.t_hours, res_rescue.cparp_pct, lw=3, linestyle=:dash, color=:forestgreen,
      label="$(rescue_name) ($(round(res_rescue.t2d_hours, digits=1)) h)")
hline!(p_ex, [10.0], linestyle=:dot, color=:gray, lw=1.5, label="10% threshold")
plot!(p_ex, xlabel="Time (h)", ylabel="Cleaved PARP (%)",
      title="Apoptosis Rescue Screen", xlims=(0, 144), ylims=(0, 70),
      xtickfontsize=11, ytickfontsize=11, xguidefontsize=13, yguidefontsize=13)
display(p_ex)

````{admonition} 💡 Hint — Rescuing BCL2 Overexpression
:class: tip, dropdown

Think about where **BCL2** sits in the pathway (`tBid → BAX → MOMP → Cyto c / SMAC → Caspase-3 → PARP`):
1. **Direct rescue (venetoclax mechanism):** Lowering mitochondrial BCL2 synthesis (`Dict("k1_88" => 0.5)`) directly cancels out the `2×` increase in `Bcl2t`!
2. **Overwhelming BCL2 upstream:** Over-expressing pro-apoptotic **BAX** (`Dict("k1_86" => 2.5)`) or **BID** (`Dict("k1_81" => 2.5)`) provides enough active `tBid`/`BAX` to saturate BCL2 and restore rapid MOMP!
3. **What about downstream XIAP (`"k1_95" => 0.05`) or Caspase-3 (`"k1_101" => 5.0`)?** Try them and see! Why is rescuing *downstream* of MOMP much less effective than rescuing *at or upstream of* the mitochondrial BCL2/BAX checkpoint?
````

````{admonition} ✅ Reveal Solution — Rescuing BCL2 Overexpression
:class: note, dropdown

Because **MOMP** is the rate-limiting all-or-nothing gatekeeper, targets that directly shift the **BCL2 : BAX / BID** balance (such as **BAX overexpression**, **BID overexpression**, or **inhibiting BCL2 synthesis** like the clinical drug **venetoclax**) dramatically rescue `BCL2 × 2.0` back to WT death timing (~60 h)—whereas downstream targets like XIAP knockdown alone cannot trigger death until MOMP has actually occurred!

```julia
test_bcl2 = 2.0
candidates = [
    ("WT",                       1.0,       Dict{String, Float64}()),
    ("BCL2 × 2 (untreated)",     test_bcl2, Dict{String, Float64}()),
    ("+ BAX OE (2.5×)",          test_bcl2, Dict("k1_86" => 2.5)),
    ("+ BID OE (2.5×)",          test_bcl2, Dict("k1_81" => 2.5)),
    ("+ BCL2 inh / venetoclax",  test_bcl2, Dict("k1_88" => 0.5)),
    ("+ XIAP KD (0.05×)",        test_bcl2, Dict("k1_95" => 0.05)),
]

results = [(name, run_apoptosis(m; mods=d)) for (name, m, d) in candidates]
display(DataFrame(
    Condition       = [x[1] for x in results],
    Time_to_Death_h = [round(x[2].t2d_hours, digits=1) for x in results],
))

p_sol = plot(xlabel="Time (h)", ylabel="Cleaved PARP (%)",
             title="Comparing Candidate Rescues for BCL2 × 2.0",
             xlims=(0, 144), ylims=(0, 70), legend=:bottomright)
for (name, r) in results
    plot!(p_sol, r.t_hours, r.cparp_pct, lw=2.8, label="$(name) ($(round(r.t2d_hours, digits=1)) h)")
end
hline!(p_sol, [10.0], linestyle=:dot, color=:gray, lw=1.2, label="10% threshold")
display(p_sol)
```
````

---

**Next:** Now let's model the **other half** of Double-Hit Lymphoma—how **c-MYC** accelerates cell division—in [**5.3 Cell Cycle Modelling: c-MYC Shortens G1 (Figure 1E)**](03-cell-cycle-myc-model.ipynb)!